# Razonamiento con incertidumbre

Las reglas de un [sistema experto](rules_and_expert_systems.ipynb) son seguras: si se cumplen las condiciones, se cumple la conclusión. La evidencia real rara vez funciona así. Un síntoma hace una enfermedad *más probable*, una alarma *sugiere* un robo. MYCIN asociaba *factores de certeza* a sus reglas, números que se combinaban de formas sin una base matemática firme. La respuesta con fundamento es la **probabilidad**: grados de creencia que se actualizan con el **teorema de Bayes** a medida que llega la evidencia ([el teorema y una prueba diagnóstica](../probability/README.md)).

El problema es el tamaño. La **distribución conjunta** completa, una probabilidad para cada combinación de valores, necesita 2ⁿ − 1 números para *n* variables de sí o no: alrededor de un millón para 20 variables. Una **red bayesiana** (Judea Pearl, 1985) evita escribirlos todos registrando solo qué variables influyen directamente en cuáles.

## Una red bayesiana

El ejemplo clásico de Pearl. Una casa tiene una alarma antirrobo, que un robo suele hacer saltar, y que a veces también hace saltar un pequeño terremoto. Dos vecinos, John y Mary, han prometido llamar cuando la oigan; John a veces confunde el teléfono con la alarma, y Mary a veces no la oye por la música alta.

```mermaid
flowchart TD
    burglary["Robo"] --> alarm["Alarma"]
    earthquake["Terremoto"] --> alarm
    alarm --> john["John llama"]
    alarm --> mary["Mary llama"]
```

Cada flecha significa *influye directamente*. Cada variable recibe una tabla de probabilidades dados solo sus padres: las llamadas dependen de la alarma, no directamente del robo. Estos son los números del libro de texto de Russell y Norvig, escritos a mano como lo haría un experto:

In [ ]:
probability_of_burglary = 0.001
probability_of_earthquake = 0.002
probability_of_alarm = {  # dado (robo, terremoto)
    (True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001,
}
probability_john_calls = {True: 0.90, False: 0.05}  # dada la alarma
probability_mary_calls = {True: 0.70, False: 0.01}  # dada la alarma

Son 10 números, donde la distribución conjunta completa de cinco variables de sí o no necesita 31. La probabilidad conjunta de cualquier situación completa es el producto de la probabilidad de cada variable dados sus padres:

In [ ]:
def probability_of(value, probability_of_true):
    return probability_of_true if value else 1 - probability_of_true


def joint_probability(burglary, earthquake, alarm, john_calls, mary_calls):
    return (probability_of(burglary, probability_of_burglary)
            * probability_of(earthquake, probability_of_earthquake)
            * probability_of(alarm, probability_of_alarm[(burglary, earthquake)])
            * probability_of(john_calls, probability_john_calls[alarm])
            * probability_of(mary_calls, probability_mary_calls[alarm]))


print(f"{joint_probability(False, False, True, True, True):.6f}",
      "= P(alarm, both call, but no burglary and no earthquake)")

0.000628 = P(alarm, both call, but no burglary and no earthquake)


## Inferencia por enumeración

Para responder una pregunta como *¿qué probabilidad hay de un robo, dado que han llamado los dos vecinos?*, se suma la probabilidad conjunta de cada situación que concuerda con la evidencia, por separado para cada respuesta, y se normaliza. Como la comprobación de modelos en [lógica](logic.ipynb), enumera todos los mundos posibles, aquí ponderando cada uno por su probabilidad.

In [ ]:
from itertools import product

variables = ["burglary", "earthquake", "alarm", "john_calls", "mary_calls"]


def probability_given(query, evidence):
    totals = {True: 0.0, False: 0.0}
    for values in product([True, False], repeat=len(variables)):
        world = dict(zip(variables, values))
        if all(world[name] == value for name, value in evidence.items()):
            totals[world[query]] += joint_probability(*values)
    return totals[True] / (totals[True] + totals[False])

In [ ]:
both_call = {"john_calls": True, "mary_calls": True}
print(f"P(burglary | both call)   = {probability_given('burglary', both_call):.3f}")
print(f"P(earthquake | both call) = {probability_given('earthquake', both_call):.3f}")
print(f"P(burglary | only John is known to call) = {probability_given('burglary', {'john_calls': True}):.3f}")

P(burglary | both call)   = 0.284
P(earthquake | both call) = 0.176
P(burglary | only John is known to call) = 0.016


Incluso con los dos vecinos llamando, un robo es menos probable que lo contrario: los robos son tan poco frecuentes (uno de cada mil) que una falsa alarma sigue siendo la explicación más probable. Es la misma lección que la [prueba diagnóstica](../probability/diagnostic_test.md) de una enfermedad poco frecuente: la probabilidad **a priori** importa tanto como la evidencia.

## *Explaining away* (una causa explica el efecto)

La evidencia a favor de una causa puede bajar la probabilidad de otra. Una vez que se sabe que la alarma ha saltado, enterarse de que hubo un terremoto hace mucho menos probable un robo, porque el terremoto ya explica la alarma:

In [ ]:
print(f"P(burglary | alarm)             = {probability_given('burglary', {'alarm': True}):.4f}")
print(f"P(burglary | alarm, earthquake) = {probability_given('burglary', {'alarm': True, 'earthquake': True}):.4f}")

P(burglary | alarm)             = 0.3736
P(burglary | alarm, earthquake) = 0.0033


Las reglas no pueden expresar esto: ninguna regla si-entonces sobre robos menciona los terremotos. La red lo obtiene gratis de la estructura de las influencias, y por eso las redes bayesianas sustituyeron a los factores de certeza en los sistemas de diagnóstico desde finales de los 80.

La enumeración crece exponencialmente con el número de variables. Los sistemas reales usan algoritmos exactos más inteligentes, como la eliminación de variables, o aproximan la respuesta por muestreo.

## Del conocimiento al aprendizaje

Las tablas de arriba se escribieron a mano, como lo haría un experto. También se pueden **estimar a partir de datos**: simula cien mil noches en este barrio, y cuenta.

In [ ]:
import random

random.seed(0)
nights = []
for _ in range(100_000):
    burglary = random.random() < probability_of_burglary
    earthquake = random.random() < probability_of_earthquake
    alarm = random.random() < probability_of_alarm[(burglary, earthquake)]
    john_calls = random.random() < probability_john_calls[alarm]
    mary_calls = random.random() < probability_mary_calls[alarm]
    nights.append((burglary, earthquake, alarm, john_calls, mary_calls))

In [ ]:
nights_with_alarm = [night for night in nights if night[2]]
print("Nights with a burglary:", sum(night[0] for night in nights))
print("Nights with an alarm:", len(nights_with_alarm))
print(f"Estimated P(John calls | alarm) = {sum(night[3] for night in nights_with_alarm) / len(nights_with_alarm):.2f} (true 0.90)")
print(f"Estimated P(Mary calls | alarm) = {sum(night[4] for night in nights_with_alarm) / len(nights_with_alarm):.2f} (true 0.70)")
print("Nights with both a burglary and an earthquake:", sum(night[0] and night[1] for night in nights))

Nights with a burglary: 111
Nights with an alarm: 258
Estimated P(John calls | alarm) = 0.87 (true 0.90)
Estimated P(Mary calls | alarm) = 0.68 (true 0.70)
Nights with both a burglary and an earthquake: 0


Contar recupera las probabilidades frecuentes, cerca de las verdaderas. Las combinaciones poco frecuentes son otra cosa: un robo durante un terremoto debería ocurrir unas dos veces en un millón de noches, y en estas cien mil no ocurrió nunca, así que su fila de la tabla de la alarma no se puede estimar en absoluto. Donde los datos escasean, los números de un experto cubren el hueco; donde abundan, contar es más barato y más preciso que preguntar.

Estimar las tablas a partir de datos es **aprendizaje automático**: la estructura de la red escrita por una persona, sus probabilidades aprendidas. El clasificador **naive Bayes** es el caso más sencillo, una red en la que la clase apunta a cada característica, con cada probabilidad contada a partir de ejemplos etiquetados; es una profundización de aprendizaje supervisado *(próximamente)*. Así que el razonamiento bayesiano no está a un lado de la línea entre IA clásica y aprendizaje automático: lo que decide el lado es de dónde vienen las probabilidades.